# Encoder Models

In the previous notebook, BERT gave the word `bank` two different vectors, depending on the sentence it appeared in. This notebook explains how that happens and what such vectors are useful for.

BERT is an **encoder**: it reads a whole text and returns one vector per token, but it never writes text. That makes it a good fit for tasks such as finding texts with the same meaning and classifying texts, sometimes without a single labeled example. In this notebook you see how BERT was trained, look at **attention**, the mechanism that brings the context into each vector, and use sentence vectors to classify product reviews.

## Learning Objectives

By the end of this notebook, you should be able to:

- Recover a masked word with BERT.
- Plot the attention weights of a token.
- Classify reviews with embeddings and zero-shot labels.

## Setup

In [ ]:
# standard library
from pathlib import Path

# data stack
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# deep learning stack
import torch

# nlp stack
import transformers
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer, pipeline

# ml stack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

In [ ]:
# Hide the hub's log output, as in notebook 01. The note about
# unauthenticated requests is printed below Python and appears once regardless.
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

## 1. Two Kinds of Language Model

Almost every model in use is one of two shapes, and they are built for opposite jobs:

- a **representation model** is **encoder-only**. It reads a whole text at once and returns vectors. BERT is the example, and so (at a much smaller scale) is a bag of words: both turn a text into numbers and neither writes anything.
- a **generative model** is **decoder-only**. It reads what has been written so far and produces the next token, one at a time. GPT and its descendants are these, and notebook 03 examines one.

The two shapes differ in one rule: which tokens a token may draw on when its vector is updated. Section 3 shows the mechanism; the rule itself fits in one picture:

<img src="../assets/attention_masks.png" alt="Two six-by-six grids for the tokens I sat on the river bank. In the encoder grid every cell is filled: each token attends to every token. In the decoder grid only the cells on and below the diagonal are filled: each token attends to itself and the tokens before it." width="800">

The split decides what a model can be asked to do. Nothing here will produce a sentence; everything here produces numbers that something else (a classifier, a search index, a clustering algorithm) then uses.

## 2. How BERT Learned: Masked Language Modeling

BERT was never shown a labeled example during pretraining. It was shown ordinary text with words hidden, and asked to guess them. That is **masked language modeling**, and it is the whole of the pretraining objective: hide about 15% of the tokens, predict each one from everything on both sides of it.

`AutoModelForMaskedLM` loads BERT with the head that does the guessing still attached, so the training task can be run by hand.

In [ ]:
masked_bert = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")
masked_bert.eval()
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def fill_mask(sentence, k=5):
    """Return the k tokens BERT considers most likely at the [MASK] in `sentence`."""
    encoded = tokenizer(sentence, return_tensors="pt")
    # where the [MASK] sits in the sequence
    position = (encoded.input_ids[0] == tokenizer.mask_token_id).nonzero()[0, 0]
    with torch.no_grad():
        logits = masked_bert(**encoded).logits[0, position]
    best = logits.softmax(-1).topk(k)
    return pd.DataFrame(
        {
            "token": [tokenizer.decode(t) for t in best.indices],
            "probability": [round(float(p), 3) for p in best.values],
        }
    )

In [ ]:
fill_mask("The capital of France is [MASK].")

**Result.** `paris` at 0.417, then four more French cities. Nobody told BERT about France; it read a great deal of text in which Paris follows that phrase, and the association was stored in the weights.

The word on both sides matters here. A model that writes sees only what came before the gap, because what comes after has not been written yet. BERT sees the whole sentence, which is what the **bi-directional** in its name means, and it is why BERT is good at judging a text and incapable of continuing one.

In [ ]:
# The ambiguous word of notebook 01, with the word itself hidden.
fill_mask("I sat on the river [MASK] and watched the water.")

**Result.** BERT's first choice is `##bank` at 0.70, and the `##` is the continuation marker of the previous notebook: joined to the token before it, this reads `riverbank`, one word. Standalone `bank` follows at 0.20. Together the two spellings take 0.90 of the probability, recovered from `river` and `water` alone.

That is the same ability measured as a cosine in the previous notebook, seen from the other side: to fill the gap correctly, the model must already combine the surrounding words into something that means *the edge of a river*. The next section watches it do so.

## 3. Attention: Where the Context Comes From

Each layer of the model rewrites every token's vector as a weighted average of all the tokens' vectors, itself included. The weights are called **attention**, they are computed from the vectors rather than fixed in advance, and they are the mechanism the previous notebook left unexplained.

Those weights can be read out, but only if the model is loaded a particular way.

In [ ]:
# `attn_implementation="eager"` is required. The faster default computes attention
# without ever materializing the weight matrix, so `output_attentions=True` returns an
# empty tuple with no warning at all.
bert = AutoModel.from_pretrained("bert-base-uncased", attn_implementation="eager")
bert.eval()

sentence = "I sat on the river bank and watched the water."
encoded = tokenizer(sentence, return_tensors="pt")

In [ ]:
with torch.no_grad():
    attentions = bert(**encoded, output_attentions=True).attentions

print(
    f"{len(attentions)} layers, each {attentions[0].shape[1]} heads, "
    f"each a {attentions[0].shape[2]}x{attentions[0].shape[3]} table"
)

**Result.** 12 layers × 12 heads, and every head holds a 13 × 13 table: for each of the 13 tokens, how much it draws on each of the 13. The heads are independent and look at different things, so the cell below averages them to read a layer as a whole.

In [ ]:
def attends_to(sentence, layer, token="bank", k=4):
    """The k tokens `token` draws on most in `layer` of `sentence`, averaged over the heads."""
    encoded = tokenizer(sentence, return_tensors="pt")
    pieces = tokenizer.convert_ids_to_tokens(encoded.input_ids[0])
    with torch.no_grad():
        attentions = bert(**encoded, output_attentions=True).attentions

    weights = attentions[layer][0].mean(0)[
        pieces.index(token)
    ]  # mean over the 12 heads
    best = weights.topk(k)
    return {pieces[i]: round(float(w), 2) for w, i in zip(best.values, best.indices)}

In [ ]:
for layer in (0, 6, 11):
    print(f"layer {layer:2d}: {attends_to(sentence, layer)}")

The cell below draws the whole table for layers 0 and 11, averaged over the heads, with the row of `bank` framed. Each row is one token being updated, each column a token it draws on, and darker means more weight. Each panel has its own colour scale.

In [ ]:
pieces = tokenizer.convert_ids_to_tokens(encoded.input_ids[0])
row = pieces.index("bank")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, layer in zip(axes, (0, 11)):
    weights = attentions[layer][0].mean(0)  # mean over the 12 heads
    image = ax.imshow(weights, cmap="Blues", vmin=0)
    fig.colorbar(image, ax=ax, shrink=0.8)
    ax.set_xticks(range(len(pieces)), pieces, rotation=90)
    ax.set_yticks(range(len(pieces)), pieces)
    ax.axhline(row - 0.5, color="darkorange")  # frame the row of `bank`
    ax.axhline(row + 0.5, color="darkorange")
    ax.set_title(f"layer {layer}")
    ax.set_xlabel("attends to")
axes[0].set_ylabel("token being updated")
plt.tight_layout()
plt.show()

**Result.** In **layer 0**, `bank` draws most on `river` (0.25), then `watched` and `water`: precisely the words that fix which bank it is. This is the mechanism, visible: the contextual vector differs between sentences because the weighted average is taken over different neighbors.

In layers 6 and 11 the weight collapses onto `[SEP]` and the full stop. That is not a bug. It is well documented, and the usual reading is that a head with nothing to contribute places its weight on a token that carries no meaning, rather than distorting the average. It is also the reason attention pictures should be read with care: a head attending to punctuation is a head doing nothing.

**Your turn:** `attends_to` takes a sentence, so the savings-bank sentence can be passed to it directly. Compare what `bank` draws on at layer 0 in each of the two sentences.

In [ ]:
money_sentence = "She works at the bank downtown and counts money."

print(f"river bank:   {attends_to(sentence, 0)}")
print(f"savings bank: {attends_to(money_sentence, 0)}")

## 4. One Vector for a Whole Sentence

Sections 2 and 3 produced one vector per *token*. Most work needs one vector per *text*: to compare two documents, to search a collection, to cluster a corpus, or to hand a text to a classifier.

Averaging BERT's token vectors does this badly. The fix is a model trained for it: a **sentence transformer** is an encoder fine-tuned so that texts meaning the same thing land close together, which plain BERT was never asked to do. `gte-small` is one, at 67 MB.

In [ ]:
# dtype=float32 is not cosmetic: the model ships as float16, which has no fast CPU
# path, and encoding runs about five times slower without this.
encoder = SentenceTransformer(
    "thenlper/gte-small", model_kwargs={"dtype": torch.float32}
)
print(
    f"{encoder.get_embedding_dimension()} numbers per text, "
    f"up to {encoder.max_seq_length} tokens of input"
)

In [ ]:
probe = [
    "The film was a complete waste of time.",
    "I regret spending two hours on that movie.",  # the same opinion, no shared content word
    "A stunning, unforgettable piece of cinema.",  # the opposite opinion, same subject
    "The central bank raised interest rates.",  # another subject entirely
]
similarity = cosine_similarity(encoder.encode(probe))
pd.DataFrame(
    np.round(similarity, 2), index=[s[:34] for s in probe], columns=range(1, 5)
)

**Result.** Read the first row. Sentence 2 is the closest at 0.89 although it shares no content word with sentence 1. `waste`, `time`, `regret`, `hours` have nothing in common, and a bag of words would score the pair zero. The opposite opinion about the same subject follows at 0.82, and the unrelated sentence comes last at 0.73.

**The absolute numbers mean little.** Nothing here falls below 0.70, including the pair with nothing in common, because models of this family pack all ordinary English into a narrow band of the space. Only the *ordering* is informative, so never set a threshold like 0.8 by intuition; measure where the cut falls on your own data.

## 5. Classifying with Embeddings

Vectors that put related texts together are already most of a classifier. The data below is 3,000 product reviews, half labeled positive and half negative, in `reviews.csv`: short ones, 28 words in the middle of the distribution.

**Only 500 are used for training.** That is the realistic case: labels are hand-made and expensive, and a few hundred is what a real project starts with. The remaining 2,500 are the test set.

In [ ]:
reviews = pd.read_csv(Path("../data") / "reviews.csv")
train, test = train_test_split(
    reviews, train_size=500, random_state=42, stratify=reviews["sentiment"]
)
print(f"{len(train)} to train on, {len(test)} to be scored on")
reviews.head(3)

First the baseline, which is the bag of words: count the words, weight them by TF-IDF, fit a logistic regression. On a long text this is a strong method and hard to beat.

In [ ]:
vectorizer = TfidfVectorizer()
baseline = LogisticRegression(max_iter=2000).fit(
    vectorizer.fit_transform(train.text), train.sentiment
)
tfidf_f1 = f1_score(
    test.sentiment, baseline.predict(vectorizer.transform(test.text)), average="macro"
)
print(f"TF-IDF + logistic regression: {tfidf_f1:.3f} macro F1")

Now the same classifier on the same labels, with the text represented by the encoder instead. Nothing in the transformer is trained. It is run once over each text and its output is stored, which is why this takes about half a minute rather than hours.

In [ ]:
# ~3,000 texts through the encoder; roughly 30 seconds on a laptop CPU.
train_vectors = encoder.encode(train.text.tolist())
test_vectors = encoder.encode(test.text.tolist())
print("one row per review, one column per dimension:", train_vectors.shape)

In [ ]:
classifier = LogisticRegression(max_iter=2000).fit(train_vectors, train.sentiment)
predictions = classifier.predict(test_vectors)
print(classification_report(test.sentiment, predictions, digits=3))

**Result.** 0.938 against the baseline's 0.779: sixteen points, on identical labels, with the same classifier on top.

The gap is this wide **because the texts are short and the labels are few**. A bag of words can only use a word it saw in training, and 500 reviews of 28 words leave most of the language unseen; a review praising something as `superb` is unreachable if only `excellent` was in the training set. The encoder was pretrained on far more text than this corpus contains, so it already knows those two words are neighbors. Give TF-IDF tens of thousands of long documents and the gap narrows. This is a statement about *this* situation, not a ranking of the two methods.

## 6. Zero-Shot: Classifying with No Labels At All

If the encoder puts texts with similar meanings close together, it will also put a text close to a *description of its class*. So the labels can be embedded like any other sentence, and each review assigned to whichever description it sits nearest. No training, and no labeled examples whatsoever.

In [ ]:
descriptions = ["A negative review", "A positive review"]
label_vectors = encoder.encode(descriptions)
class_names = np.array(["negative", "positive"])  # in the same order as `descriptions`

# For each review, which of the two descriptions is its vector closest to?
nearest = cosine_similarity(test_vectors, label_vectors).argmax(axis=1)
zero_shot_f1 = f1_score(test.sentiment, class_names[nearest], average="macro")
print(f"zero-shot: {zero_shot_f1:.3f} macro F1, from zero labels")

**Result.** 0.912, which beats TF-IDF trained on 500 labeled reviews by thirteen points while using none.

That is the single most useful fact about pretrained embeddings, and much of why modern NLP relies on them so heavily. The cost moved: instead of labeling data you write a sentence describing each class, and most of the remaining accuracy comes from writing those sentences well. The exercise below shows how much this matters.

**Your turn:** the descriptions are a free choice and they matter. Try at least three pairs: a bare pair of words such as `bad` and `good`, a pair of full sentences in the voice of a reviewer, and one of your own. Score each. One of them loses six points against the pair above.

In [ ]:
candidates = [
    ["bad", "good"],
    ["This product disappointed me.", "This product delighted me."],
    ["A negative review", "A positive review"],
]

for pair in candidates:
    nearest = cosine_similarity(test_vectors, encoder.encode(pair)).argmax(axis=1)
    score = f1_score(test.sentiment, class_names[nearest], average="macro")
    print(f"{score:.3f}  {pair}")

## 7. Fine-Tuning, and Why Nothing Here Trained a Model

Every model in this notebook arrived already knowing something, and none of them was trained here. That is the normal way to use one, and the options can be ordered by cost:

- **Pretraining** is section 2's masked language modeling: no labels, enormous quantities of text, and the expensive one by a wide margin. It produces a model that knows the language and no task. Someone else does this, once.
- **Fine-tuning** continues training a pretrained model on labeled data for one task, usually with a new output head. Thousands of examples, and minutes to hours.
- **Embeddings plus a classifier**, as in section 5: the encoder is frozen, each text is encoded once, and only the logistic regression is fitted. No training run at all.
- **Zero-shot**, as in section 6: nothing is fitted, and the labels are written rather than collected.

Which to choose is mostly a question of how many labels exist, and it is worth starting with the cheapest option: fine-tune only once the frozen encoder's vectors have been shown not to separate the classes. Where someone has already fine-tuned a model for the task, use theirs.

In [ ]:
# A model someone else fine-tuned on labeled film reviews, 256 MB. DistilBERT is BERT
# distilled to 6 layers from 12, carrying a two-class head where section 2's had the one
# that guesses masked words.
sentiment = pipeline(
    "sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english"
)

for text in test.text.head(3):
    result = sentiment(text)[0]
    print(f"{result['label']:8s} {result['score']:.2f}  {text[:60]}...")

**Result.** One line, a label and a confidence. Note what it does *not* give you: a vector. A fine-tuned classifier answers one question and nothing else, where section 4's encoder produces a representation that a classifier, a search index or a clustering algorithm can all use. That generality is why the embedding is the thing worth computing, and it is what clustering, topic modeling and semantic search are all built on.

## Summary

- Language models come in two shapes. A **representation model** is encoder-only, reads a whole text at once and returns vectors; a **generative model** is decoder-only and writes one token at a time. BERT is the first kind and produces no text ever.
- BERT was pretrained by **masked language modeling**: hide a word, predict it from both sides. That is where `paris` after *the capital of France is* comes from, and no labels were involved.
- **Attention** is the mechanism: each layer rewrites a token's vector as a weighted average over all the tokens, with weights computed from the vectors. In layer 0, `bank` draws on `river`. Reading the weights out requires `attn_implementation="eager"`.
- Later layers concentrate their attention on `[SEP]` and punctuation. Attention pictures are evidence about one head in one layer, not a window into the model's reasoning.
- A **sentence transformer** gives one vector per text. Only the ordering of the cosines is meaningful; the absolute values all sit high.
- On short texts with few labels, **embeddings plus a logistic regression** beat TF-IDF by sixteen points, and **zero-shot against embedded label descriptions** beat it by thirteen while using no labels at all. How the descriptions are worded is then the thing that decides the accuracy.

## Sources

- The split into representation and generative models is chapter 1 of *Hands-On Large Language Models* by Jay Alammar and Maarten Grootendorst (O'Reilly, 2024). Sections 4 to 6 follow chapter 4's *Classification Tasks That Leverage Embeddings*, including its zero-shot method and its `A negative review` / `A positive review` descriptions; section 7 condenses chapter 11. The book's [companion code](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) is Apache-2.0.
- The book classifies Rotten Tomatoes reviews, which carry no redistributable license. This repository uses Amazon Polarity instead, for the same reason the book's choice works: the reviews are short.
- `bert-base-uncased`, `thenlper/gte-small` and `distilbert-base-uncased-finetuned-sst-2-english` come from the [Hugging Face model hub](https://huggingface.co/).
- On attention settling into `[SEP]`: Clark, K., Khandelwal, U., Levy, O. and Manning, C. D. (2019), [What Does BERT Look At? An Analysis of BERT's Attention](https://arxiv.org/abs/1906.04341).

The repository [README](README.md) lists every source and license.